# Organization Entity Resolution with Splink

This notebook implements a robust entity resolution strategy for clinical trial organizations using Splink.

## Goals
1. Match duplicate organization records
2. Minimize false positives (key requirement from previous attempts)
3. Provide evaluation metrics to measure improvement
4. Create a testing strategy for model validation

## Strategy Overview
- Use multi-level fuzzy matching on organization names
- Extract additional features from metadata for auxiliary matching
- Apply strict blocking rules to reduce false positive space
- Use high match thresholds (≥0.85) to prioritize precision
- Leverage term frequency adjustments for common vs uncommon names
- Evaluate using both edge metrics and cluster analysis

## 1. Setup and Data Loading

In [ ]:
# Install required packages
# !pip install splink pandas duckdb altair

In [ ]:
import pandas as pd
import json
from splink import DuckDBAPI, Linker, SettingsCreator, splink_datasets
import splink.duckdb.comparison_library as cl
import splink.duckdb.comparison_level_library as cll
import splink.duckdb.blocking_rule_library as brl
from splink.exploratory import profile_columns
from splink.evaluation import prediction_errors_from_labels_table
import warnings
warnings.filterwarnings('ignore')

print("✓ Libraries imported successfully")

In [ ]:
# Load data
df = pd.read_csv('prod_test/data/test_data.csv')

print(f"Loaded {len(df)} records")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

## 2. Data Preparation and Feature Engineering

In [ ]:
# Parse metadata JSON and extract useful fields
def parse_metadata(metadata_str):
    """Parse metadata string and extract key fields"""
    try:
        # Convert Java-style properties to JSON
        metadata_str = metadata_str.replace('=', ':').replace(', ', ',').replace('{', '{"').replace('}', '"}').replace(':', '":"').replace(',', '","')
        # This is tricky - the metadata is not valid JSON, it's more like Java properties
        # Let's use a simpler regex-based approach
        pass
    except:
        pass
    return {}

# Alternative: manually extract fields using regex
import re

def extract_metadata_field(metadata_str, field_name):
    """Extract a specific field from metadata string"""
    pattern = f"{field_name}=([^,}}]+)"
    match = re.search(pattern, metadata_str)
    if match:
        return match.group(1).strip()
    return None

# Extract useful fields
df['name_normalized'] = df['metadata'].apply(lambda x: extract_metadata_field(x, 'name_normalized'))
df['name_prefix_5'] = df['metadata'].apply(lambda x: extract_metadata_field(x, 'name_prefix_5'))
df['name_prefix_10'] = df['metadata'].apply(lambda x: extract_metadata_field(x, 'name_prefix_10'))
df['contact_type'] = df['metadata'].apply(lambda x: extract_metadata_field(x, 'contact_type'))

# Create additional features for matching
# Extract potential university name (often comes after "of" or ",")
def extract_university(name):
    """Extract university name from organization name"""
    if not isinstance(name, str):
        return None
    
    # Look for patterns like "of X University" or ", X University"
    patterns = [
        r'of ([^,]+University[^,]*)',
        r', ([^,]+University[^,]*)$',
        r'Affiliated to ([^,]+)$'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, name)
        if match:
            return match.group(1).strip()
    return None

df['university_name'] = df['name'].apply(extract_university)

# Extract hospital type (First, Second, Third, etc.)
def extract_hospital_order(name):
    """Extract if it's First, Second, Third hospital"""
    if not isinstance(name, str):
        return None
    
    for order in ['First', 'Second', 'Third', 'Fourth', 'Fifth', 'Sixth', 'Seventh', 'Eighth', 'Ninth']:
        if order in name:
            return order
    return None

df['hospital_order'] = df['name'].apply(extract_hospital_order)

# Create a clean name for matching (use name_normalized if available, else lowercase)
df['name_clean'] = df['name_normalized'].fillna(df['name'].str.lower())

# Add unique ID for tracking
df['unique_id'] = df.index

print(f"\nExtracted features:")
print(f"- name_normalized: {df['name_normalized'].notna().sum()} non-null")
print(f"- university_name: {df['university_name'].notna().sum()} non-null")
print(f"- hospital_order: {df['hospital_order'].notna().sum()} non-null")

# Show examples
print("\nExample extracted data:")
df[['name', 'name_clean', 'university_name', 'hospital_order']].head(10)

## 3. Exploratory Data Analysis

In [ ]:
# Analyze name patterns
print("Top 20 most common organizations by name_count:")
df.nlargest(20, 'name_count')[['name', 'name_count']].to_string()

In [ ]:
# Find potential duplicates manually (for validation)
# Group by similar names
print("\nPotential duplicates (same university name):")
university_counts = df[df['university_name'].notna()].groupby('university_name').size()
print(f"Universities appearing multiple times: {(university_counts > 1).sum()}")
print("\nExamples:")
for univ, count in university_counts[university_counts > 1].head(5).items():
    print(f"\n{univ} ({count} occurrences):")
    matching_orgs = df[df['university_name'] == univ][['name', 'hospital_order']]
    print(matching_orgs.to_string())

In [ ]:
# Profile columns for Splink
db_api = DuckDBAPI()

# Select columns for Splink
splink_df = df[[
    'unique_id',
    'mismatch_id',
    'name',
    'name_clean',
    'name_prefix_5',
    'name_prefix_10',
    'university_name',
    'hospital_order',
    'name_count'
]].copy()

# Profile columns to understand data quality
profile = profile_columns(
    splink_df,
    db_api=db_api,
    column_expressions=['name_clean', 'university_name', 'hospital_order']
)
profile

## 4. Create Testing Strategy

To measure improvement and reduce false positives, we'll create:
1. **Manual labeled pairs** - A small set of known matches and non-matches
2. **Synthetic test cases** - Edge cases to test specific scenarios
3. **Cluster validation** - Analyze cluster quality metrics

In [ ]:
# Create manual labels for testing
# These are examples - you should expand this with your domain knowledge

labeled_pairs = [
    # TRUE MATCHES - Different representations of same organization
    {
        'unique_id_l': None,  # Will be filled based on finding these names
        'unique_id_r': None,
        'name_l': 'Zhongshan Hospital, Fudan University',
        'name_r': 'Zhongshan Hospital Affiliated to Fudan University',
        'clerical_match_score': 1
    },
    {
        'unique_id_l': None,
        'unique_id_r': None,
        'name_l': 'Huashan Hospital, Fudan University',
        'name_r': 'Huashan Hospital Affiliated to Fudan University',
        'clerical_match_score': 1
    },
    # FALSE MATCHES - Different organizations with similar names
    {
        'unique_id_l': None,
        'unique_id_r': None,
        'name_l': 'The First Affiliated Hospital of Zhengzhou University',
        'name_r': 'The First Affiliated Hospital of Chongqing Medical University',
        'clerical_match_score': 0
    },
    {
        'unique_id_l': None,
        'unique_id_r': None,
        'name_l': 'The First Affiliated Hospital of Anhui Medical University',
        'name_r': 'The First Affiliated Hospital of Guangxi Medical University',
        'clerical_match_score': 0
    },
    {
        'unique_id_l': None,
        'unique_id_r': None,
        'name_l': 'The First Affiliated Hospital of Sun Yat-sen University',
        'name_r': 'The First Affiliated Hospital of Sun Yat-Sen University',  # Just capitalization difference
        'clerical_match_score': 1
    },
]

# Helper function to find record IDs by name
def find_id_by_name(name):
    matches = splink_df[splink_df['name'] == name]
    if len(matches) > 0:
        return matches.iloc[0]['unique_id']
    return None

# Fill in the IDs
for pair in labeled_pairs:
    pair['unique_id_l'] = find_id_by_name(pair['name_l'])
    pair['unique_id_r'] = find_id_by_name(pair['name_r'])

# Create labels dataframe
labels_df = pd.DataFrame(labeled_pairs)
labels_df = labels_df[labels_df['unique_id_l'].notna() & labels_df['unique_id_r'].notna()]

print(f"Created {len(labels_df)} labeled pairs:")
print(f"- Matches: {(labels_df['clerical_match_score'] == 1).sum()}")
print(f"- Non-matches: {(labels_df['clerical_match_score'] == 0).sum()}")
print("\nLabeled pairs:")
labels_df[['name_l', 'name_r', 'clerical_match_score']]

## 5. Configure Splink Model

Key design decisions to reduce false positives:
1. **High Jaro-Winkler thresholds** (0.90-0.95) for name matching
2. **Multiple comparison levels** to distinguish between strong and weak matches
3. **Term frequency adjustments** to weight uncommon names higher
4. **Strict blocking rules** using exact prefix matching
5. **High match threshold** (≥0.85) for final predictions

In [ ]:
# Configure Splink settings
settings = SettingsCreator(
    link_type="dedupe_only",  # Finding duplicates within single dataset
    
    # Comparisons - multi-level fuzzy matching
    comparisons=[
        # Primary comparison: normalized name
        cl.CustomComparison(
            output_column_name="name_clean",
            comparison_levels=[
                cll.NullLevel("name_clean"),
                # Exact match - highest confidence
                cll.ExactMatchLevel(
                    "name_clean",
                    term_frequency_adjustments=True,
                    label_for_charts="Exact match"
                ),
                # Very high similarity - likely same org with minor differences
                cll.JaroWinklerLevel(
                    "name_clean",
                    0.95,
                    term_frequency_adjustments=True,
                    label_for_charts="Jaro-Winkler >= 0.95"
                ),
                # High similarity - probably same org
                cll.JaroWinklerLevel(
                    "name_clean",
                    0.92,
                    term_frequency_adjustments=True,
                    label_for_charts="Jaro-Winkler >= 0.92"
                ),
                # Moderate similarity - needs additional evidence
                cll.JaroWinklerLevel(
                    "name_clean",
                    0.88,
                    label_for_charts="Jaro-Winkler >= 0.88"
                ),
                # Low similarity - likely different but capture for analysis
                cll.JaroWinklerLevel(
                    "name_clean",
                    0.85,
                    label_for_charts="Jaro-Winkler >= 0.85"
                ),
                # Everything else
                cll.ElseLevel()
            ]
        ),
        
        # Supporting comparison: university name
        cl.CustomComparison(
            output_column_name="university_name",
            comparison_levels=[
                cll.NullLevel("university_name"),
                cll.ExactMatchLevel(
                    "university_name",
                    term_frequency_adjustments=True,
                    label_for_charts="University exact match"
                ),
                cll.JaroWinklerLevel(
                    "university_name",
                    0.90,
                    label_for_charts="University Jaro-Winkler >= 0.90"
                ),
                cll.ElseLevel()
            ]
        ),
        
        # Supporting comparison: hospital order (First, Second, etc.)
        cl.ExactMatch(
            "hospital_order",
            term_frequency_adjustments=True
        ).configure(
            label_for_charts="Hospital order"
        ),
    ],
    
    # Blocking rules - these determine which pairs to compare
    # Use multiple rules to ensure we don't miss matches
    blocking_rules_to_generate_predictions=[
        # Block 1: First 15 characters of name must match exactly
        "substr(l.name_clean, 1, 15) = substr(r.name_clean, 1, 15)",
        
        # Block 2: Same university name (exact)
        "l.university_name = r.university_name",
        
        # Block 3: Same 10-char prefix AND same hospital order
        "substr(l.name_clean, 1, 10) = substr(r.name_clean, 1, 10) and l.hospital_order = r.hospital_order",
        
        # Block 4: Very similar university name (for fuzzy cases)
        brl.block_on(
            "university_name",
            salting_partitions=1
        ),
    ],
    
    # Additional settings
    retain_matching_columns=True,
    retain_intermediate_calculation_columns=True,
)

print("✓ Splink settings configured")

## 6. Create Linker and Train Model

In [ ]:
# Create linker
linker = Linker(
    splink_df,
    settings,
    db_api=db_api
)

print("✓ Linker created successfully")

In [ ]:
# View deterministic rules (if any exact matches)
linker.count_num_comparisons_from_blocking_rules_for_prediction(
    splink_df
)

In [ ]:
# Estimate probability two random records match (prior)
linker.estimate_probability_two_random_records_match(
    "substr(l.name_clean, 1, 10) = substr(r.name_clean, 1, 10)",
    recall=0.7
)

print("✓ Estimated prior probability")

In [ ]:
# Train model using Expectation Maximization (unsupervised)
# We'll train on different blocking rules to get good parameter estimates

print("Training on name_clean comparison...")
training_session_names = linker.estimate_parameters_using_expectation_maximisation(
    "substr(l.name_clean, 1, 12) = substr(r.name_clean, 1, 12)",
    comparisons_to_deactivate=["university_name", "hospital_order"]
)

print("\nTraining on university_name comparison...")
training_session_uni = linker.estimate_parameters_using_expectation_maximisation(
    "l.university_name = r.university_name",
    comparisons_to_deactivate=["name_clean", "hospital_order"]
)

print("\n✓ Model training complete")

In [ ]:
# View match weights chart
linker.match_weights_chart()

## 7. Generate Predictions

We'll use a **high threshold (0.85)** to minimize false positives.

In [ ]:
# Generate predictions with high threshold
MATCH_THRESHOLD = 0.85  # Adjust this to tune precision/recall trade-off

df_predictions = linker.predict(
    threshold_match_probability=MATCH_THRESHOLD
)

# Convert to pandas for analysis
predictions_df = df_predictions.as_pandas_dataframe()

print(f"✓ Generated {len(predictions_df)} predictions at threshold {MATCH_THRESHOLD}")
print(f"\nMatch probability distribution:")
print(predictions_df['match_probability'].describe())

In [ ]:
# View sample predictions
print("Sample high-confidence matches:")
sample_predictions = predictions_df.nlargest(20, 'match_probability')[[
    'name_l', 'name_r', 'match_probability', 'match_weight'
]]
sample_predictions

In [ ]:
# Interactive waterfall chart for specific pairs
# Pick a prediction to examine in detail
linker.waterfall_chart(
    predictions_df.to_dict('records'),
    filter_nulls=False
)

## 8. Cluster Analysis

In [ ]:
# Create clusters from predictions
clusters = linker.cluster_pairwise_predictions_at_threshold(
    df_predictions,
    threshold_match_probability=MATCH_THRESHOLD
)

clusters_df = clusters.as_pandas_dataframe()

print(f"✓ Created {clusters_df['cluster_id'].nunique()} clusters")
print(f"\nCluster size distribution:")
cluster_sizes = clusters_df.groupby('cluster_id').size()
print(cluster_sizes.value_counts().sort_index())

In [ ]:
# Analyze large clusters (potential false positive groups)
print("Clusters with 3+ members (review for false positives):")
large_clusters = cluster_sizes[cluster_sizes >= 3]

for cluster_id in large_clusters.head(10).index:
    cluster_members = clusters_df[clusters_df['cluster_id'] == cluster_id]
    print(f"\n--- Cluster {cluster_id} ({len(cluster_members)} members) ---")
    print(cluster_members[['name', 'university_name', 'hospital_order']].to_string())

In [ ]:
# Cluster metrics
from splink.cluster_metrics import cluster_metrics

metrics = cluster_metrics(
    linker,
    df_predictions,
    threshold_match_probability=MATCH_THRESHOLD
)

metrics_df = metrics.as_pandas_dataframe()

print("Cluster quality metrics:")
print(metrics_df[[
    'cluster_id', 'n_nodes', 'n_edges', 'density', 'cluster_centralisation'
]].head(20))

In [ ]:
# Identify suspicious clusters
# Low density or high centralization may indicate false positives
suspicious_clusters = metrics_df[
    (metrics_df['density'] < 0.5) & (metrics_df['n_nodes'] >= 3)
].sort_values('density')

print(f"Found {len(suspicious_clusters)} suspicious clusters (low density, 3+ nodes)")
print("\nThese clusters may contain false positives:")
suspicious_clusters[['cluster_id', 'n_nodes', 'n_edges', 'density', 'cluster_centralisation']]

In [ ]:
# Review suspicious clusters in detail
for cluster_id in suspicious_clusters['cluster_id'].head(5):
    cluster_members = clusters_df[clusters_df['cluster_id'] == cluster_id]
    print(f"\n=== SUSPICIOUS Cluster {cluster_id} ===")
    print(cluster_members[['name', 'university_name']].to_string())
    
    # Show the edges (pairwise matches) in this cluster
    cluster_edges = predictions_df[
        predictions_df['unique_id_l'].isin(cluster_members['unique_id']) &
        predictions_df['unique_id_r'].isin(cluster_members['unique_id'])
    ]
    print(f"\nEdges in cluster:")
    print(cluster_edges[['name_l', 'name_r', 'match_probability']].to_string())

## 9. Evaluation with Labeled Data

In [ ]:
# Evaluate against labeled data (if we have any)
if len(labels_df) > 0:
    from splink.evaluation import accuracy_analysis_from_labels_table
    
    # The labels need to be in the database
    db_api.register_table(labels_df, "labels")
    
    # Run accuracy analysis
    accuracy_chart = accuracy_analysis_from_labels_table(
        linker,
        "labels",
        match_weight_round_to_nearest=0.1
    )
    
    accuracy_chart
else:
    print("No labeled data available for evaluation")
    print("To improve model validation, add more labeled pairs to the testing section")

In [ ]:
# Prediction errors analysis (if we have labels)
if len(labels_df) > 0:
    errors = prediction_errors_from_labels_table(
        linker,
        "labels",
        include_false_positives=True,
        include_false_negatives=True
    )
    
    errors_df = errors.as_pandas_dataframe()
    
    print(f"False Positives: {len(errors_df[errors_df['truth'] == 'FP'])}")
    print(f"False Negatives: {len(errors_df[errors_df['truth'] == 'FN'])}")
    print(f"True Positives: {len(errors_df[errors_df['truth'] == 'TP'])}")
    print(f"True Negatives: {len(errors_df[errors_df['truth'] == 'TN'])}")
    
    # Show errors
    if len(errors_df[errors_df['truth'] == 'FP']) > 0:
        print("\nFalse Positives (incorrectly matched):")
        print(errors_df[errors_df['truth'] == 'FP'][['name_l', 'name_r', 'match_probability']])
    
    if len(errors_df[errors_df['truth'] == 'FN']) > 0:
        print("\nFalse Negatives (missed matches):")
        print(errors_df[errors_df['truth'] == 'FN'][['name_l', 'name_r', 'match_probability']])

## 10. Threshold Tuning

Test different thresholds to find the optimal balance

In [ ]:
# Test multiple thresholds
thresholds_to_test = [0.75, 0.80, 0.85, 0.90, 0.95]

threshold_results = []

for threshold in thresholds_to_test:
    preds_at_threshold = predictions_df[predictions_df['match_probability'] >= threshold]
    clusters_at_threshold = linker.cluster_pairwise_predictions_at_threshold(
        df_predictions,
        threshold_match_probability=threshold
    ).as_pandas_dataframe()
    
    n_pairs = len(preds_at_threshold)
    n_clusters = clusters_at_threshold['cluster_id'].nunique()
    n_records_in_clusters = len(clusters_at_threshold)
    
    threshold_results.append({
        'threshold': threshold,
        'n_pairs_matched': n_pairs,
        'n_clusters': n_clusters,
        'n_records_in_clusters': n_records_in_clusters,
        'pct_records_matched': f"{100 * n_records_in_clusters / len(splink_df):.1f}%"
    })

threshold_comparison = pd.DataFrame(threshold_results)
print("\nThreshold comparison:")
threshold_comparison

## 11. Export Results

In [ ]:
# Export final clusters
clusters_df['original_name'] = clusters_df['unique_id'].map(
    dict(zip(splink_df['unique_id'], splink_df['name']))
)

# Save results
clusters_df.to_csv('prod_test/data/matched_clusters.csv', index=False)
predictions_df.to_csv('prod_test/data/match_predictions.csv', index=False)
threshold_comparison.to_csv('prod_test/data/threshold_comparison.csv', index=False)

print("✓ Results exported to:")
print("  - prod_test/data/matched_clusters.csv")
print("  - prod_test/data/match_predictions.csv")
print("  - prod_test/data/threshold_comparison.csv")

## 12. Summary and Recommendations

### Key Metrics for Measuring Improvement

1. **Precision** (minimize false positives):
   - Review large clusters with low density
   - Manually validate random sample of matches
   - Track false positive rate if you have labeled data

2. **Recall** (don't miss true matches):
   - Compare against known duplicates
   - Look for singleton clusters that should be merged

3. **Cluster Quality**:
   - **Density**: Higher is better (>0.7 ideal)
   - **Centralization**: Lower is better (<0.5 ideal)
   - Large clusters with low density are suspicious

### Strategies to Reduce False Positives

If you're still seeing too many false positives:

1. **Increase match threshold** (try 0.90 or 0.95)
2. **Tighten blocking rules** (require longer exact prefix matches)
3. **Add more comparison levels** with exact matches on key distinguishing terms
4. **Extract more features**:
   - City/location information
   - Hospital type keywords
   - External IDs if available
5. **Manual rules** for known problematic patterns

### Testing Strategy

1. **Expand labeled data**: Add 50-100 manually labeled pairs
   - Focus on edge cases (similar names, different orgs)
   - Include various hospital types and naming patterns

2. **Cluster sampling**: Randomly sample clusters and manually validate

3. **A/B testing**: Compare model versions on held-out labeled set

4. **Production monitoring**: Track cluster sizes and densities over time

### Next Steps

1. Review the suspicious clusters identified above
2. Add more labeled pairs based on your domain knowledge
3. Tune the threshold using the comparison table
4. Consider adding geographic or other metadata if available
5. Iterate on blocking rules and comparison levels